# CVPR S1 bước 2: selftest, train codec latent đa rate, eval trên Kodak

**Cài đặt:** Accelerator = **GPU T4 x1**, Internet = **On**. **Add Input → Notebook Output** của `s1_cache`. Chạy bằng **Save Version → Save & Run All (Commit)** để notebook chạy nền; mỗi bước tự bỏ qua phần đã làm, nên hết giờ thì chạy lại là tiếp tục.

1. `selftest`: model ngẫu nhiên, nén trên GPU và giải nén trên CPU. Phải ra **PASS** thì notebook mới chạy tiếp.
2. `train`: tối đa 9.5 giờ, tự lưu `s1_out/last.pt` mỗi 5k bước. Nếu bị ngắt, chạy lại sẽ tiếp tục từ checkpoint (nhớ gắn Output của lần trước làm Input và chép `last.pt` vào `s1_out/`).
3. `eval`: 24 ảnh Kodak × 8 mức rate, bitstream thật, so với trần DC-AE.

**Kết quả:** `s1_results.zip` (log, summary, csv và checkpoint), đặt vào `results/s1/`.

*Sinh bởi `notebooks/build.py` từ commit `7e391c4` của repo; đừng sửa tay notebook này.*

In [ ]:
import socket
for host in ('pypi.org', 'huggingface.co'):
    try:
        socket.create_connection((host, 443), timeout=5).close()
    except OSError:
        raise RuntimeError(f'Không có Internet ({host}): bật Settings → Internet → On '
                           '(Kaggle yêu cầu tài khoản đã xác minh số điện thoại)')
print('Internet OK')

In [ ]:
!mkdir -p experiments/s1 src/ratflow src/ratflow/codec src/ratflow/entropy src/ratflow/eval src/ratflow/nn

In [ ]:
%%writefile src/ratflow/__init__.py
"""Rate-as-Time flow-bridge image codec (CVPR 2027) and its CLIC 2027 submission."""


In [ ]:
%%writefile src/ratflow/codec/__init__.py
"""Codec stages: S1 latent codec (latent_codec.py); the flow bridge (S2/S3) comes next."""


In [ ]:
%%writefile src/ratflow/codec/latent_codec.py
"""S1: variable-rate hyperprior codec on DC-AE f32c32 latents, with subtractive dither in the bitstream.

    latent (B, 32, h, w), already multiplied by the DC-AE scaling factor
      g_a ->  y (B, M, h/s, w/s)             s = y_stride (1 or 2)
      y_r = y * gain[r]                      quantisation step 1/gain[r] per channel (rate level r)
      h_a(y_r) -> z (B, Z, h/4s, w/4s)       z_hat = round(z), DiscretePrior
      h_s(z_hat; r) [integer net] -> (scale index, mu)   (sigma, mu) during training
      y_r_hat = q - u,  q = round(y_r + u)   shared dither u (Prop. 1): error ~ U(-1/2, 1/2), independent of y
      y_hat = y_r_hat / gain[r];  g_s -> latent_hat  (trained with MSE, so latent_hat ~ E[latent | bitstream])

Everything that decides a CDF (h_s, tables, dither, centre) is integer/bit-exact; g_a, h_a, g_s are float.
`compress` / `decompress` produce and parse a self-contained byte string per image.
"""
from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn

from ..entropy import rans
from ..entropy.intnet import IntSequential, QConv2d, Up2
from ..entropy.models import DiscretePrior, DitheredGaussian, ScaleMeanHead, gaussian_bits

MAGIC = 0x52  # 'R'
VERSION = 1


class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, 1, 1)
        self.c2 = nn.Conv2d(c, c, 3, 1, 1)
        self.act = nn.GELU()

    def forward(self, x):
        return x + self.c2(self.act(self.c1(x)))


class LatentCodec(nn.Module):
    def __init__(self, latent_ch: int = 32, N: int = 192, M: int = 128, Z: int = 96, n_rates: int = 8,
                 y_stride: int = 2, n_offsets: int = 16, mu_frac: int = 4, gain_range=(0.25, 8.0)):
        super().__init__()
        assert y_stride in (1, 2)
        self.config = dict(latent_ch=latent_ch, N=N, M=M, Z=Z, n_rates=n_rates, y_stride=y_stride,
                           n_offsets=n_offsets, mu_frac=mu_frac, gain_range=list(gain_range))
        self.y_stride, self.mu_frac, self.n_rates = y_stride, mu_frac, n_rates
        self.multiple = 4 * y_stride  # latent H, W are padded to this

        down = [nn.Conv2d(N, N, 3, 2, 1)] if y_stride == 2 else []
        up = [nn.Conv2d(N, 4 * N, 3, 1, 1), nn.PixelShuffle(2)] if y_stride == 2 else []
        self.g_a = nn.Sequential(nn.Conv2d(latent_ch, N, 3, 1, 1), ResBlock(N), ResBlock(N), *down,
                                 ResBlock(N), ResBlock(N), nn.Conv2d(N, M, 3, 1, 1))
        self.g_s = nn.Sequential(nn.Conv2d(M, N, 3, 1, 1), ResBlock(N), ResBlock(N), *up,
                                 ResBlock(N), ResBlock(N), nn.Conv2d(N, latent_ch, 3, 1, 1))
        self.h_a = nn.Sequential(nn.Conv2d(M, N, 3, 1, 1), nn.LeakyReLU(0.1), nn.Conv2d(N, N, 5, 2, 2),
                                 nn.LeakyReLU(0.1), nn.Conv2d(N, Z, 5, 2, 2))
        self.h_s = IntSequential(
            Up2(), QConv2d(Z, N, 5, in_frac=0, out_frac=7, n_cond=n_rates),
            Up2(), QConv2d(N, N, 5, in_frac=7, out_frac=7, n_cond=n_rates),
            QConv2d(N, 2 * M, 3, in_frac=7, act=None, n_cond=n_rates),
        )
        self.head = ScaleMeanHead(self.h_s[-1].acc_frac, mu_frac)
        self.z_prior = DiscretePrior(Z)
        self.y_model = DitheredGaussian(n_offsets)
        g = torch.exp(torch.linspace(math.log(gain_range[0]), math.log(gain_range[1]), n_rates))
        self.log_gain = nn.Parameter(torch.log(g)[:, None].repeat(1, M))  # (n_rates, M)

    # ------------------------------------------------------------------ helpers
    def _pad(self, lat):
        h, w = lat.shape[-2:]
        m = self.multiple
        ph, pw = (-h) % m, (-w) % m
        if ph or pw:
            lat = F.pad(lat, (0, pw, 0, ph), mode="reflect" if min(h, w) > max(ph, pw) else "replicate")
        return lat, (h, w)

    def gain(self, r: torch.Tensor):
        return torch.exp(self.log_gain[r])[:, :, None, None]

    # ------------------------------------------------------------------ training / evaluation forward
    def forward(self, lat: torch.Tensor, r: torch.Tensor) -> dict:
        """lat: (B, 32, h, w) scaled latent; r: (B,) rate levels. Bits are exact code lengths of q given u
        (up to table quantisation), so bpp is comparable with real bitstreams."""
        x, (h, w) = self._pad(lat)
        g = self.gain(r)
        y_r = self.g_a(x) * g
        z_hat, bits_z = self.z_prior(self.h_a(y_r))
        sigma, mu = self.head(self.h_s(z_hat, r))
        y_r_hat = DitheredGaussian.quantize(y_r)
        bits_y = gaussian_bits(y_r_hat, sigma, mu)
        lat_hat = self.g_s(y_r_hat / g)[..., :h, :w]
        return dict(lat_hat=lat_hat, bits_y=bits_y.flatten(1).sum(1), bits_z=bits_z.flatten(1).sum(1),
                    y_r=y_r, sigma=sigma)

    # ------------------------------------------------------------------ bitstream
    @torch.no_grad()
    def compress(self, lat: torch.Tensor, r: int, seed: int = 0) -> bytes:
        """lat: (1, 32, h, w) scaled latent. Returns the full byte string for one image."""
        assert lat.shape[0] == 1
        x, (h, w) = self._pad(lat)
        rr = torch.tensor([r], device=lat.device)
        y_r = self.g_a(x) * self.gain(rr)
        z_int = torch.round(self.h_a(y_r))
        zb = self.z_prior.compress(z_int)
        idx, mu_int = self._coding_params(z_int, r)
        yb = self.y_model.compress(y_r, idx, mu_int, self.mu_frac, seed)
        out = bytearray([MAGIC, VERSION])
        for v in (h, w, r, seed, len(zb)):
            rans.put_varint(int(v), out)
        return bytes(out + zb + yb)

    @torch.no_grad()
    def decompress(self, blob: bytes, device="cpu") -> torch.Tensor:
        assert blob[0] == MAGIC and blob[1] == VERSION, "not a ratflow S1 stream"
        pos, hdr = 2, []
        for _ in range(5):
            v, pos = rans.get_varint(blob, pos)
            hdr.append(v)
        h, w, r, seed, nz = hdr
        m = self.multiple
        H, W = (h + (-h) % m) // self.y_stride, (w + (-w) % m) // self.y_stride
        z_shape = (1, self.config["Z"], H // 4, W // 4)
        z_int = self.z_prior.decompress(blob[pos:pos + nz], z_shape)
        idx, mu_int = self._coding_params(z_int, r)
        y_r_hat = self.y_model.decompress(blob[pos + nz:], idx, mu_int, self.mu_frac, seed).to(device)
        rr = torch.tensor([r], device=device)
        return self.g_s(y_r_hat / self.gain(rr).to(device))[..., :h, :w]

    def _coding_params(self, z_int: torch.Tensor, r: int):
        """Integer path on CPU (bit-exact on every machine)."""
        acc = self.h_s.forward_int(z_int.double().cpu(), torch.tensor([r]))
        return self.head.coding_params(acc)

    @torch.no_grad()
    def prepare_coding(self):
        """Freeze the learned z prior into integer tables. Call before compress/decompress and before saving."""
        self.z_prior.update_tables()
        return self


In [ ]:
%%writefile src/ratflow/entropy/__init__.py
"""Deterministic entropy coding: numpy rANS, integer CDF tables, integer hyper-networks."""


In [ ]:
%%writefile src/ratflow/entropy/coding.py
"""Value-level coding on top of rANS: integers per element, table per element, escape side stream.

    blob = encode_values(q, table_idx, cdf, half)       # q: integer values, table_idx: which table each uses
    q = decode_values(blob, table_idx, cdf, half, inv)

Numpy only. Blob layout: varint(len(rans)) | rans | varint(n_escapes) | zigzag varints.
"""
from __future__ import annotations

import numpy as np

from . import rans


def encode_values(q: np.ndarray, t: np.ndarray, cdf: np.ndarray, half: np.ndarray) -> bytes:
    q = np.asarray(q, np.int64).ravel()
    t = np.asarray(t, np.int64).ravel()
    k = half[t]
    esc = np.abs(q) > k
    sym = np.where(esc, 2 * k + 1, q + k)
    body = rans.encode(sym, t, cdf)
    out = bytearray()
    rans.put_varint(len(body), out)
    out += body
    ev = q[esc]
    rans.put_varint(ev.size, out)
    for v in ev.tolist():
        rans.put_varint(rans.zigzag(int(v)), out)
    return bytes(out)


def decode_values(blob: bytes, t: np.ndarray, cdf: np.ndarray, half: np.ndarray, inv: np.ndarray):
    t = np.asarray(t, np.int64).ravel()
    n_body, pos = rans.get_varint(blob, 0)
    sym = rans.decode(blob[pos:pos + n_body], t, cdf, inv)
    pos += n_body
    k = half[t]
    q = sym - k
    esc = sym == 2 * k + 1
    n_esc, pos = rans.get_varint(blob, pos)
    if n_esc != int(esc.sum()):
        raise ValueError("escape count mismatch")
    vals = []
    for _ in range(n_esc):
        u, pos = rans.get_varint(blob, pos)
        vals.append(rans.unzigzag(u))
    q[esc] = np.asarray(vals, np.int64)
    if pos != len(blob):
        raise ValueError("trailing bytes in blob")
    return q


In [ ]:
%%writefile src/ratflow/entropy/dither.py
"""Shared subtractive dither (Proposition 1) with bit-exact integer bookkeeping (numpy only).

The dither u ~ U(-1/2, 1/2) is drawn from PCG64.random_raw: raw bit-generator output is fully
specified by the PCG64 algorithm, so encoder and decoder get the same bits on any machine and
numpy version. u has 16 fractional bits: u = (u16 + 1/2) / 2**16 - 1/2, exactly representable.

Coding (in units of the quantisation step, i.e. on the gain-scaled latent y_r):
  encoder: q = round(y_r + u);  decoder: y_hat_r = q - u  ->  y_hat_r - y_r ~ U(-1/2, 1/2), independent of y.
  entropy model: q ~ N(mu + u, sigma^2) discretised; with c = mu + u, the symbol is v = q - floor(c)
  and the table is chosen by (scale index, floor(frac(c) * n_offsets)).
mu comes from the integer hyper-network with MU_FRAC fractional bits, so c is computed exactly in
units of 2**-17 (u needs 17 bits once the 1/2 offset is included).
"""
from __future__ import annotations

import numpy as np

U_BITS = 16
C_BITS = U_BITS + 1


def dither_u16(n: int, seed: int) -> np.ndarray:
    """n raw 16-bit draws (int64 in [0, 2**16)), identical on every machine for a given seed."""
    raw = np.random.PCG64(seed).random_raw(n)  # uint64
    return (raw >> np.uint64(64 - U_BITS)).astype(np.int64)


def u_float(u16: np.ndarray) -> np.ndarray:
    """Exact dyadic value of u in (-1/2, 1/2)."""
    return (u16.astype(np.float64) + 0.5) / 2 ** U_BITS - 0.5


def centre_c17(mu_int: np.ndarray, u16: np.ndarray, mu_frac: int) -> np.ndarray:
    """c = mu + u in units of 2**-17, as int64."""
    assert mu_frac <= C_BITS
    return mu_int.astype(np.int64) * (1 << (C_BITS - mu_frac)) + 2 * u16 + 1 - (1 << U_BITS)


def table_and_base(c17: np.ndarray, scale_idx: np.ndarray, n_offsets: int):
    """(table index, base = floor(c)) for each element; symbol value v = q - base."""
    base = np.floor_divide(c17, 1 << C_BITS)
    frac = c17 - base * (1 << C_BITS)  # in [0, 2**17)
    off = (frac * n_offsets) >> C_BITS
    return scale_idx.astype(np.int64) * n_offsets + off, base


In [ ]:
%%writefile src/ratflow/entropy/intnet.py
"""Integer networks for entropy parameters (Ballé et al., ICLR 2019, simplified to power-of-two scales).

Anything that decides a CDF (hyper-synthesis h_s, context models) must give bit-identical outputs on
the encoder machine and on the CLIC server. A0 showed float32/bf16 differ between GPU and CPU (~80
indices per 8M), while "integers carried in float64" match everywhere: every product and partial sum
is an integer below 2**53, so the result does not depend on summation order or kernel choice.

Fixed-point convention: a tensor with `frac` fractional bits stores round-down(v * 2**frac) as integers.
- QConv2d weights: int8 with W fractional bits; bias: integer at (W + in_frac) fractional bits.
- Hidden activations (ReLU): out = clamp(floor(acc / 2**shift), 0, 2**a_bits - 1),
  shift = W + in_frac - out_frac (must be >= 0).
- Last layer (act=None) returns the raw accumulator with (W + in_frac) fractional bits.

Training uses the same quantisers with straight-through gradients (`forward`); decoding uses
`forward_int` on integer inputs. Run `forward_int` on CPU (default) to avoid any GPU kernel choice.
"""
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn


def ste(x_q: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Value of x_q, gradient of x."""
    return x + (x_q - x).detach()


class QConv2d(nn.Conv2d):
    """Integer conv. With n_cond > 0 it also has a per-condition bias (e.g. one per rate level),
    selected by `cond` (LongTensor (B,)); the bias stays an integer, so decoding stays exact."""

    def __init__(self, cin, cout, k, stride=1, *, in_frac: int, out_frac: int | None = None, act: str | None = "relu",
                 w_frac: int = 7, w_max: int = 127, a_bits: int = 16, groups: int = 1, n_cond: int = 0):
        super().__init__(cin, cout, k, stride, padding=k // 2, groups=groups, bias=True)
        self.in_frac, self.w_frac, self.w_max, self.a_bits, self.act = in_frac, w_frac, w_max, a_bits, act
        self.out_frac = out_frac if act else w_frac + in_frac
        assert act in (None, "relu") and self.w_frac + self.in_frac >= self.out_frac
        self.cond_bias = nn.Parameter(torch.zeros(n_cond, cout)) if n_cond else None

    @property
    def acc_frac(self) -> int:
        return self.w_frac + self.in_frac

    def _wq(self):
        return torch.clamp(torch.round(self.weight * 2 ** self.w_frac), -self.w_max, self.w_max)

    def _bias(self, cond, batch):
        b = self.bias.expand(batch, -1)
        if self.cond_bias is not None:
            assert cond is not None, "this layer needs a condition index"
            b = b + self.cond_bias[cond.to(self.cond_bias.device)]
        return b  # (B, cout), float

    def forward(self, x, cond=None):  # float in/out, quantisers with STE
        w = ste(self._wq() / 2 ** self.w_frac, self.weight)
        b = self._bias(cond, x.shape[0])
        b = ste(torch.round(b * 2 ** self.acc_frac) / 2 ** self.acc_frac, b)
        y = F.conv2d(x, w, None, self.stride, self.padding, 1, self.groups) + b[:, :, None, None]
        if self.act is None:
            return y
        y = F.relu(y)
        q = torch.clamp(torch.floor(y * 2 ** self.out_frac), 0, 2 ** self.a_bits - 1) / 2 ** self.out_frac
        return ste(q, y)

    @torch.no_grad()
    def forward_int(self, x_int: torch.Tensor, cond=None) -> torch.Tensor:  # float64 tensors holding integers
        dev = x_int.device
        w = self._wq().to(torch.float64).to(dev)
        b = torch.round(self._bias(cond, x_int.shape[0]).double() * 2 ** self.acc_frac).to(dev)
        acc = F.conv2d(x_int.to(torch.float64), w, None, self.stride, self.padding, 1, self.groups) + b[:, :, None, None]
        if self.act is None:
            return acc
        shift = self.acc_frac - self.out_frac
        return torch.clamp(torch.floor(F.relu(acc) / 2 ** shift), 0, 2 ** self.a_bits - 1)


class Up2(nn.Module):
    """Nearest-neighbour x2 upsampling (exact on integers)."""

    def forward(self, x, cond=None):
        return F.interpolate(x, scale_factor=2, mode="nearest")

    forward_int = forward


class IntSequential(nn.Sequential):
    """Sequential that passes an optional condition index to every layer."""

    def forward(self, x, cond=None):
        for m in self:
            x = m(x, cond)
        return x

    def forward_int(self, x_int: torch.Tensor, cond=None) -> torch.Tensor:
        for m in self:
            x_int = m.forward_int(x_int, cond)
        return x_int

    @property
    def out_frac(self) -> int:
        return [m for m in self if isinstance(m, QConv2d)][-1].out_frac


def check_exact_range(net: IntSequential, in_max: float) -> float:
    """Worst-case |accumulator| bound; must stay < 2**53 for the float64 trick to be exact."""
    bound, worst = in_max, 0.0
    for m in net:
        if isinstance(m, QConv2d):
            fan_in = m.in_channels // m.groups * m.kernel_size[0] * m.kernel_size[1]
            acc = bound * m.w_max * fan_in + 2 ** (m.acc_frac + 16)
            worst = max(worst, acc)
            bound = 2 ** m.a_bits - 1 if m.act else acc
    assert worst < 2 ** 53, f"accumulator bound {worst:.3g} >= 2**53"
    return worst


In [ ]:
%%writefile src/ratflow/entropy/models.py
"""Entropy models with bit-exact decoding.

- DiscretePrior: learned per-channel pmf for the hyper-latent z (factorised, integer tables).
- ScaleMeanHead: turns the integer accumulator of an IntSequential into (scale index, dyadic mean).
- GaussianConditional: rate for training; compress/decompress y given (scale index, mean).
- DitheredGaussian: the same with shared subtractive dither (Proposition 1) in the bitstream.

CDF tables are module buffers. They are built from floats when the module is created or when
`update_tables()` runs, and must then be loaded from the SAME checkpoint on the encoder and the
decoder side (never rebuilt on the decoder), so libm differences cannot change them.
"""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from . import coding, dither, tables
from .intnet import ste

LN2 = math.log(2.0)


def _np(t: torch.Tensor) -> np.ndarray:
    return t.detach().cpu().numpy()


class _Tables(nn.Module):
    """Holds (cdf, sym_half) as persistent buffers and a lazily built inverse table."""

    def _set_tables(self, cdf: np.ndarray, half: np.ndarray):
        self.register_buffer("cdf", torch.from_numpy(cdf.astype(np.int64)), persistent=True)
        self.register_buffer("sym_half", torch.from_numpy(half.astype(np.int64)), persistent=True)  # not "half": nn.Module.half()
        self._inv = None

    def _load_from_state_dict(self, *args, **kwargs):  # tables come from the checkpoint: drop the cache
        super()._load_from_state_dict(*args, **kwargs)
        self._inv = None

    use_inverse = True  # inverse table: 128 KB per table; off -> binary-search decoding

    def np_tables(self):
        cdf, half = _np(self.cdf).astype(np.uint32), _np(self.sym_half)
        if self.use_inverse and self._inv is None:
            self._inv = tables.inverse(cdf)
        return cdf, half, self._inv if self.use_inverse else None


class DiscretePrior(_Tables):
    """p(z_c = v) = softmax(logits_c)[v + half], v in [-half, half], plus an escape bin.

    Training: z is rounded with a straight-through estimator, so the rate trains the logits (and the
    rest of the model through z_hat) but gives no gradient w.r.t. z itself. The hyper-rate is small."""

    def __init__(self, channels: int, half: int = 32, init_scale: float = 4.0):
        super().__init__()
        self.channels, self.half_width = channels, half
        v = torch.arange(-half, half + 1, dtype=torch.float32)
        base = torch.cat([-(v / init_scale) ** 2 / 2, torch.tensor([-10.0])])
        self.logits = nn.Parameter(base.repeat(channels, 1))
        self._set_tables(np.zeros((channels, 2 * half + 3), np.int64), np.full(channels, half, np.int64))

    def forward(self, z: torch.Tensor):
        """Returns (z_hat, bits per element)."""
        z_hat = ste(torch.round(z), z)
        q = torch.round(z).detach().long()
        esc = q.abs() > self.half_width
        idx = torch.where(esc, torch.full_like(q, 2 * self.half_width + 1), q + self.half_width)
        logp = F.log_softmax(self.logits, dim=1)  # (C, W)
        C = z.shape[1]
        lp = logp[torch.arange(C, device=z.device).view(1, C, 1, 1).expand_as(idx), idx]
        extra = torch.where(esc, 2 * torch.log2(q.abs().float() + 1) + 1, torch.zeros_like(z))  # varint-ish
        return z_hat, -lp / LN2 + extra

    @torch.no_grad()
    def update_tables(self):
        pmf = _np(F.softmax(self.logits.double(), dim=1))
        self._set_tables(*tables.pmf_tables(pmf))

    def compress(self, z: torch.Tensor) -> bytes:
        cdf, half, _ = self.np_tables()
        q = _np(torch.round(z)).astype(np.int64)
        t = np.broadcast_to(np.arange(q.shape[1]).reshape(1, -1, 1, 1), q.shape)
        return coding.encode_values(q, t, cdf, half)

    def decompress(self, blob: bytes, shape) -> torch.Tensor:
        cdf, half, inv = self.np_tables()
        t = np.broadcast_to(np.arange(shape[1]).reshape(1, -1, 1, 1), shape)
        return torch.from_numpy(coding.decode_values(blob, t, cdf, half, inv).reshape(shape)).float()


class ScaleMeanHead(nn.Module):
    """Maps the last-layer accumulator (2*M channels: scale part, mean part) of an IntSequential to
    sigma (training) or an integer scale index + a dyadic mean with MU_FRAC fractional bits (coding)."""

    def __init__(self, acc_frac: int, mu_frac: int = 4):
        super().__init__()
        self.acc_frac, self.mu_frac = acc_frac, mu_frac
        s = tables.scale_table()
        # idx = #{j >= 1 : sigma(acc) >= s_j}, sigma(a) = s_0 + softplus(a)  =>  a >= log(expm1(s_j - s_0))
        thr = np.ceil(np.log(np.expm1(s[1:] - s[0])) * 2 ** acc_frac).astype(np.int64)
        self.register_buffer("thr", torch.from_numpy(thr), persistent=True)
        self.s0 = float(s[0])

    def forward(self, acc: torch.Tensor):
        """Training path. Returns (sigma, mu); mu is floored to the MU_FRAC grid with STE."""
        a_s, a_m = acc.chunk(2, dim=1)
        sigma = self.s0 + F.softplus(a_s)
        mu = ste(torch.floor(a_m * 2 ** self.mu_frac) / 2 ** self.mu_frac, a_m)
        return sigma, mu

    @torch.no_grad()
    def coding_params(self, acc_int: torch.Tensor):
        """acc_int: integer accumulator (float64). Returns (scale index int64, mu_int int64)."""
        a_s, a_m = acc_int.chunk(2, dim=1)
        idx = torch.bucketize(a_s, self.thr.to(a_s.device).to(torch.float64), right=True)
        mu_int = torch.floor(a_m / 2 ** (self.acc_frac - self.mu_frac))
        return idx.long().cpu(), mu_int.long().cpu()


class GaussianConditional(_Tables):
    def __init__(self):
        super().__init__()
        self._set_tables(*tables.gaussian_tables())

    @staticmethod
    def bits(y: torch.Tensor, sigma: torch.Tensor, mu: torch.Tensor, training: bool):
        """Returns (y_hat, bits per element). Training: additive uniform noise; else rounding around mu (STE)."""
        if training:
            y_hat = y + torch.empty_like(y).uniform_(-0.5, 0.5)
        else:
            y_hat = ste(torch.round(y - mu) + mu, y)
        v = (y_hat - mu).abs()
        s = sigma.clamp_min(tables.SCALE_MIN)
        c = lambda x: 0.5 * torch.erfc(-x / math.sqrt(2.0))  # noqa: E731
        p = c((0.5 - v) / s) - c((-0.5 - v) / s)
        return y_hat, -torch.log2(p.clamp_min(1e-9))

    def compress(self, y: torch.Tensor, idx: torch.Tensor, mu_int: torch.Tensor, mu_frac: int) -> bytes:
        cdf, half, _ = self.np_tables()
        mu = _np(mu_int).astype(np.float64) / 2 ** mu_frac
        q = np.round(_np(y).astype(np.float64) - mu).astype(np.int64)
        return coding.encode_values(q, _np(idx), cdf, half)

    def decompress(self, blob: bytes, idx: torch.Tensor, mu_int: torch.Tensor, mu_frac: int):
        """Returns (y_hat float32, y_hat_int = y_hat * 2**mu_frac as int64 for integer context nets)."""
        cdf, half, inv = self.np_tables()
        q = coding.decode_values(blob, _np(idx), cdf, half, inv).reshape(tuple(idx.shape))
        y_int = q * (1 << mu_frac) + _np(mu_int)
        return torch.from_numpy(y_int / 2 ** mu_frac).float(), torch.from_numpy(y_int)


def gaussian_bits(y_tilde: torch.Tensor, sigma: torch.Tensor, mu: torch.Tensor) -> torch.Tensor:
    """-log2 P(bin around y_tilde) for N(mu, sigma^2). With subtractive dither and y_tilde = q - u this is
    the exact code length of q given u, not an approximation (Prop. 1 / universal quantisation)."""
    v = (y_tilde - mu).abs()
    s = sigma.clamp_min(tables.SCALE_MIN)
    c = lambda x: 0.5 * torch.erfc(-x / math.sqrt(2.0))  # noqa: E731
    return -torch.log2((c((0.5 - v) / s) - c((-0.5 - v) / s)).clamp_min(1e-9))


class DitheredGaussian(_Tables):
    """y_r (already divided by the quantisation step) coded as q = round(y_r + u), decoded as y_r_hat = q - u,
    with u the shared dither of `entropy.dither`. Tables are indexed by (scale index, frac(mu + u))."""

    use_inverse = False

    def __init__(self, n_offsets: int = 16):
        super().__init__()
        self.n_offsets = n_offsets
        self._set_tables(*tables.gaussian_tables(n_offsets=n_offsets))

    @staticmethod
    def quantize(y_r: torch.Tensor) -> torch.Tensor:
        """Training/eval forward with real (random) dither: returns q - u with straight-through gradient."""
        u = torch.rand_like(y_r) - 0.5
        return ste(torch.round(y_r + u) - u, y_r)

    def _layout(self, idx, mu_int, mu_frac, seed):
        u16 = dither.dither_u16(int(np.prod(idx.shape)), seed)
        t, base = dither.table_and_base(dither.centre_c17(_np(mu_int).ravel(), u16, mu_frac), _np(idx).ravel(),
                                        self.n_offsets)
        return u16, t, base

    def compress(self, y_r: torch.Tensor, idx: torch.Tensor, mu_int: torch.Tensor, mu_frac: int, seed: int) -> bytes:
        cdf, half, _ = self.np_tables()
        u16, t, base = self._layout(idx, mu_int, mu_frac, seed)
        q = np.round(_np(y_r).astype(np.float64).ravel() + dither.u_float(u16)).astype(np.int64)
        return coding.encode_values(q - base, t, cdf, half)

    def decompress(self, blob: bytes, idx: torch.Tensor, mu_int: torch.Tensor, mu_frac: int, seed: int) -> torch.Tensor:
        cdf, half, inv = self.np_tables()
        u16, t, base = self._layout(idx, mu_int, mu_frac, seed)
        q = coding.decode_values(blob, t, cdf, half, inv) + base
        return torch.from_numpy((q - dither.u_float(u16)).reshape(tuple(idx.shape))).float()


In [ ]:
%%writefile src/ratflow/entropy/rans.py
"""Interleaved rANS in numpy: integer-only, so bit-exact on every machine.

Each symbol i is coded with its own table t[i] from a shared set of quantised CDFs
(`cdf[t, s]`, totals 2**PREC). Symbols are spread round-robin over `lanes_for(n)` independent rANS
states, so encode/decode vectorise over lanes (one numpy step per lane-group of symbols).

State x in [L, L << 16) with L = 2**16; renormalisation moves exactly one 16-bit word, because
PREC <= 16. Stream: [u32 final state per lane][u16 words].

Tables must be built once (float -> int) and then shipped as integers; see `tables.py`.
"""
from __future__ import annotations

import numpy as np

PREC = 16
MAX_LANES = 64
SYMS_PER_LANE = 2048


def lanes_for(n: int) -> int:
    """Number of interleaved states for n symbols. Each lane costs 4 bytes of final state, which matters
    at 0.01-0.05 bpp (a Kodak image is ~1.5 KB), so small streams use few lanes; the decoder derives the
    same number from n. At most SYMS_PER_LANE steps keeps numpy decoding fast."""
    return max(1, min(MAX_LANES, -(-n // SYMS_PER_LANE)))
L = 1 << 16
MASK = (1 << PREC) - 1


def encode(symbols: np.ndarray, tables: np.ndarray, cdf: np.ndarray, n_lanes: int | None = None) -> bytes:
    """symbols[i] in [0, n_sym(tables[i])); cdf: (n_tables, max_sym + 1) uint32, cdf[:, 0] = 0, row total 2**PREC."""
    s = np.asarray(symbols, np.int64).ravel()
    t = np.asarray(tables, np.int64).ravel()
    n = s.size
    n_lanes = n_lanes or lanes_for(n)
    start = cdf[t, s].astype(np.uint64)
    freq = (cdf[t, s + 1] - cdf[t, s]).astype(np.uint64)
    if n and freq.min() == 0:
        raise ValueError("symbol with zero frequency")
    steps = -(-n // n_lanes)
    x = np.full(n_lanes, L, np.uint64)
    words = []  # one chunk per step, steps visited backwards; chunks are reversed at the end
    for k in range(steps - 1, -1, -1):
        lo, hi = k * n_lanes, min(n, (k + 1) * n_lanes)
        m = hi - lo
        xs, st, fr = x[:m], start[lo:hi], freq[lo:hi]
        x_max = ((L >> PREC) << 16) * fr
        ren = xs >= x_max
        if ren.any():
            words.append((xs[ren] & 0xFFFF).astype(np.uint16))  # ascending lanes = decoder read order
            xs[ren] >>= 16
        x[:m] = (xs // fr << PREC) + xs % fr + st
    body = np.concatenate(words[::-1]) if words else np.zeros(0, np.uint16)
    return x.astype("<u4").tobytes() + body.astype("<u2").tobytes()


def _search(cdf: np.ndarray, t: np.ndarray, slot: np.ndarray) -> np.ndarray:
    """Vectorised binary search: largest s with cdf[t, s] <= slot (no inverse table needed)."""
    lo = np.zeros(t.size, np.int64)
    hi = np.full(t.size, cdf.shape[1] - 2, np.int64)
    while True:
        open_ = lo < hi
        if not open_.any():
            return lo
        mid = (lo + hi + 1) >> 1
        ok = cdf[t, mid] <= slot
        lo = np.where(open_ & ok, mid, lo)
        hi = np.where(open_ & ~ok, mid - 1, hi)


def decode(data: bytes, tables: np.ndarray, cdf: np.ndarray, inv: np.ndarray | None = None,
           n_lanes: int | None = None) -> np.ndarray:
    """inv[t, slot] = symbol s with cdf[t, s] <= slot < cdf[t, s+1] (see `tables.inverse`);
    with inv=None a binary search over cdf is used instead (memory independent of the number of tables)."""
    t = np.asarray(tables, np.int64).ravel()
    n = t.size
    n_lanes = n_lanes or lanes_for(n)
    x = np.frombuffer(data[:4 * n_lanes], "<u4").astype(np.uint64)
    words = np.frombuffer(data[4 * n_lanes:], "<u2").astype(np.uint64)
    out = np.empty(n, np.int64)
    pos = 0
    for k in range(-(-n // n_lanes)):
        lo, hi = k * n_lanes, min(n, (k + 1) * n_lanes)
        m = hi - lo
        xs, tt = x[:m], t[lo:hi]
        slot = xs & MASK
        s = inv[tt, slot].astype(np.int64) if inv is not None else _search(cdf, tt, slot.astype(np.int64))
        st = cdf[tt, s].astype(np.uint64)
        fr = (cdf[tt, s + 1] - cdf[tt, s]).astype(np.uint64)
        xs = fr * (xs >> PREC) + slot - st
        ren = xs < L
        r = int(ren.sum())
        if r:
            xs[ren] = (xs[ren] << 16) | words[pos:pos + r]
            pos += r
        x[:m] = xs
        out[lo:hi] = s
    if pos != words.size or not np.all(x == L):  # every lane must return to the initial state
        raise ValueError("rANS integrity check failed (corrupt stream or wrong tables)")
    return out


# ---------------------------------------------------------------- small helpers for side information
def put_varint(v: int, out: bytearray):
    while True:
        b = v & 0x7F
        v >>= 7
        out.append(b | (0x80 if v else 0))
        if not v:
            return


def get_varint(buf: bytes, pos: int):
    v = shift = 0
    while True:
        b = buf[pos]
        pos += 1
        v |= (b & 0x7F) << shift
        shift += 7
        if not b & 0x80:
            return v, pos


def zigzag(v: int) -> int:
    return (v << 1) ^ (v >> 63) if v < 0 else v << 1


def unzigzag(u: int) -> int:
    return (u >> 1) ^ -(u & 1)


In [ ]:
%%writefile src/ratflow/entropy/tables.py
"""Quantised CDF tables for rANS (numpy only).

Tables are computed ONCE from floats at export time and then stored as integers in the model
file; encoder and decoder load the same integers, so libm differences (erf, exp) never matter.

Layout of a table set: cdf (n_tables, width + 1) uint32 with cdf[:, 0] = 0 and total 2**PREC;
symbol j of table t covers value j - half[t] for j < 2*half[t] + 1; symbol 2*half[t] + 1 is the
escape (value outside [-half, half], sent as a varint side stream). Unused columns have zero width.
"""
from __future__ import annotations

import math

import numpy as np

from .rans import PREC

TOTAL = 1 << PREC
SCALE_MIN, SCALE_MAX, N_SCALES = 0.11, 256.0, 64
TAIL_SIGMAS = 6.0
HALF_MAX = 1023


def scale_table() -> np.ndarray:
    """Log-spaced Gaussian scales (CompressAI-style). Index 0 is SCALE_MIN."""
    return np.exp(np.linspace(math.log(SCALE_MIN), math.log(SCALE_MAX), N_SCALES))


def quantize_pmf(p: np.ndarray, total: int = TOTAL) -> np.ndarray:
    """Float pmf -> integer frequencies >= 1 summing to `total` (deterministic)."""
    p = np.asarray(p, np.float64)
    p = p / p.sum()
    n = p.size
    f = np.floor(p * (total - n)).astype(np.int64) + 1
    f[int(np.argmax(f))] += total - int(f.sum())
    assert f.min() >= 1 and f.sum() == total
    return f


def _phi(x: np.ndarray) -> np.ndarray:
    return 0.5 * np.vectorize(math.erfc)(-x / math.sqrt(2.0))


def gaussian_tables(scales=None, n_offsets: int = 1):
    """Discretised Gaussians. Table t = scale_idx * n_offsets + offset_idx models v = q - floor(center)
    with the Gaussian centred at o = (offset_idx + 0.5) / n_offsets in [0, 1) (o = 0 if n_offsets == 1,
    i.e. zero-mean tables). Returns (cdf, half)."""
    scales = scale_table() if scales is None else np.asarray(scales, np.float64)
    offs = np.zeros(1) if n_offsets == 1 else (np.arange(n_offsets) + 0.5) / n_offsets
    half_s = np.minimum(np.ceil(TAIL_SIGMAS * scales).astype(np.int64) + (n_offsets > 1), HALF_MAX)
    half = np.repeat(half_s, len(offs))
    width = int(2 * half.max() + 2)
    cdf = np.full((len(half), width + 1), TOTAL, np.uint32)
    cdf[:, 0] = 0
    for i, (s, k) in enumerate(zip(scales, half_s)):
        v = np.arange(-k, k + 1, dtype=np.float64)
        for j, o in enumerate(offs):
            p = _phi((v + 0.5 - o) / s) - _phi((v - 0.5 - o) / s)
            tail = _phi((-k - 0.5 - o) / s) + (1.0 - _phi((k + 0.5 - o) / s))
            f = quantize_pmf(np.append(p, max(tail, 1e-12)))
            cdf[i * len(offs) + j, 1:f.size + 1] = np.cumsum(f)
    return cdf, half


def pmf_tables(pmfs: np.ndarray):
    """Learned discrete priors: pmfs (n_tables, 2*half + 2) incl. escape as last entry. Returns (cdf, half)."""
    pmfs = np.asarray(pmfs, np.float64)
    n, w = pmfs.shape
    half = np.full(n, (w - 2) // 2, np.int64)
    cdf = np.zeros((n, w + 1), np.uint32)
    for t in range(n):
        cdf[t, 1:] = np.cumsum(quantize_pmf(pmfs[t]))
    return cdf, half


def inverse(cdf: np.ndarray) -> np.ndarray:
    """inv[t, slot] = symbol whose interval contains slot (uint16; zero-width columns never hit)."""
    n, w1 = cdf.shape
    inv = np.empty((n, TOTAL), np.uint16)
    for t in range(n):
        freq = np.diff(cdf[t].astype(np.int64))
        inv[t] = np.repeat(np.arange(w1 - 1, dtype=np.uint16), freq)
    return inv


In [ ]:
%%writefile src/ratflow/eval/__init__.py


In [ ]:
%%writefile src/ratflow/eval/metrics.py
"""Shared evaluation code: every number in the paper and in the CLIC analyses goes through here,
so ceilings, baselines and our codec stay comparable.

Conventions
- Images are HxWx3 uint8 numpy arrays.
- PSNR per image from the RGB MSE. Dataset-level "CLIC PSNR" pools MSE weighted by pixel count.
- LPIPS (alex, inputs in [-1, 1]) and DISTS are computed on 512x512 tiles, weighted by tile area
  (tiles with a side < 64 are skipped), so 2K+ images fit in GPU memory.
- MS-SSIM on the full image (NaN if the short side is < 161, the minimum for 5 scales).
- FID/KID on 256x256 patches, HiFiC protocol: a grid plus a grid shifted by 128 in both directions.
- OCR CER: easyocr on the same crops (boxes detected on the original) for reference and reconstruction.

torch, lpips, piq, torchmetrics and easyocr are imported lazily so the numpy helpers work without them.
"""
from __future__ import annotations

import csv
import math
from pathlib import Path

import numpy as np
from PIL import Image

TILE = 512
FID_PATCH = 256
IMG_EXTS = (".png", ".jpg", ".jpeg")


# ---------------------------------------------------------------- io
def load(p) -> np.ndarray:
    return np.asarray(Image.open(p).convert("RGB"))


def save(a: np.ndarray, p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(a).save(p)


def images(d: Path, exts=IMG_EXTS):
    """Sorted image files under d, skipping macOS zip debris (__MACOSX/, ._*)."""
    return sorted(p for p in Path(d).rglob("*") if p.suffix.lower() in exts
                  and not p.name.startswith("._") and "__MACOSX" not in p.parts)


def write_csv(rows, path):
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, keys)
        w.writeheader()
        w.writerows(rows)


def read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))


# ---------------------------------------------------------------- torch helpers
def device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def to_tensor(a: np.ndarray, dev):
    """HxWx3 uint8 -> 1x3xHxW float in [0, 1]."""
    import torch
    return torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1)[None].float().div(255).to(dev)


# ---------------------------------------------------------------- fidelity
def mse(a: np.ndarray, b: np.ndarray) -> float:
    return float(((a.astype(np.float64) - b) ** 2).mean())


def psnr_from_mse(m: float) -> float:
    return 10 * math.log10(255 ** 2 / max(m, 1e-10))


def psnr(a, b) -> float:
    return psnr_from_mse(mse(a, b))


def pooled_psnr(mses, pixels) -> float:
    """CLIC-style dataset PSNR: MSE averaged with pixel-count weights."""
    return psnr_from_mse(sum(m * n for m, n in zip(mses, pixels)) / sum(pixels))


def msssim(a, b, dev) -> float:
    import piq
    import torch
    if min(a.shape[:2]) < 161:
        return float("nan")
    with torch.no_grad():
        return float(piq.multi_scale_ssim(to_tensor(a, dev), to_tensor(b, dev), data_range=1.0))


# ---------------------------------------------------------------- perceptual
def tiled(fn, a, b, dev, tile=TILE) -> float:
    """Area-weighted mean of fn(tile_a, tile_b) over tile x tile crops; fn takes [0,1] tensors."""
    import torch
    H, W = a.shape[:2]
    tot = wsum = 0.0
    for y in range(0, H, tile):
        for x in range(0, W, tile):
            pa, pb = a[y:y + tile, x:x + tile], b[y:y + tile, x:x + tile]
            if min(pa.shape[:2]) < 64:
                continue
            n = pa.shape[0] * pa.shape[1]
            with torch.no_grad():
                tot += float(fn(to_tensor(pa, dev), to_tensor(pb, dev))) * n
            wsum += n
    return tot / wsum


class Perceptual:
    """LPIPS-alex and DISTS with the tiling convention above."""

    def __init__(self, dev=None):
        import lpips
        import piq
        self.dev = dev or device()
        self._lpips = lpips.LPIPS(net="alex", verbose=False).to(self.dev)
        self._dists = piq.DISTS().to(self.dev)

    def lpips(self, a, b) -> float:
        return tiled(lambda x, y: self._lpips(x * 2 - 1, y * 2 - 1).mean(), a, b, self.dev)

    def dists(self, a, b) -> float:
        return tiled(lambda x, y: self._dists(x, y), a, b, self.dev)

    def all(self, a, b) -> dict:
        m = mse(a, b)
        return dict(mse=m, psnr=psnr_from_mse(m), msssim=msssim(a, b, self.dev),
                    lpips=self.lpips(a, b), dists=self.dists(a, b))


# ---------------------------------------------------------------- FID / KID
def patches(a: np.ndarray, size=FID_PATCH):
    """HiFiC protocol: size x size grid, plus the grid shifted by size // 2."""
    H, W = a.shape[:2]
    for off in (0, size // 2):
        for y in range(off, H - size + 1, size):
            for x in range(off, W - size + 1, size):
                yield a[y:y + size, x:x + size]


def fid_kid(pairs, dev=None, batch=64, kid_subset=1000) -> dict:
    """pairs: iterable of (original, reconstruction) uint8 arrays. Returns fid, kid, kid_std, n_patches."""
    import torch
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.kid import KernelInceptionDistance
    dev = dev or device()
    f = FrechetInceptionDistance(feature=2048, normalize=False).to(dev)
    k = KernelInceptionDistance(subset_size=kid_subset, normalize=False).to(dev)

    def push(ps, real):
        t = torch.from_numpy(np.stack(ps)).permute(0, 3, 1, 2).to(dev)  # uint8 NCHW
        f.update(t, real=real)
        k.update(t, real=real)

    n = 0
    for x, y in pairs:
        pr, pf = list(patches(x)), list(patches(y))
        for i in range(0, len(pr), batch):
            push(pr[i:i + batch], True)
            push(pf[i:i + batch], False)
        n += len(pr)
    km, ks = k.compute()
    return dict(fid=float(f.compute()), kid=float(km), kid_std=float(ks), n_patches=n)


# ---------------------------------------------------------------- text legibility (OCR)
def edit_distance(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def ocr_reader():
    import easyocr
    import torch
    return easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)


def box_xyxy(bbox, H, W, pad=4):
    """easyocr quadrilateral -> padded, clipped axis-aligned box (x0, y0, x1, y1)."""
    xs = [p[0] for p in bbox]
    ys = [p[1] for p in bbox]
    return (max(0, int(min(xs)) - pad), max(0, int(min(ys)) - pad),
            min(W, int(max(xs)) + pad), min(H, int(max(ys)) + pad))


def detect_text(reader, img, min_conf=0.4, min_len=2):
    """[(box_xyxy, text)] for confident detections on the original image."""
    H, W = img.shape[:2]
    return [(box_xyxy(b, H, W), t) for b, t, c in reader.readtext(img) if c > min_conf and len(t.strip()) >= min_len]


def read_boxes(reader, img, boxes):
    return ["".join(t for _, t, _ in reader.readtext(img[y0:y1, x0:x1])) for x0, y0, x1, y1 in boxes]


def cer(refs, hyps) -> float:
    """Character error rate pooled over boxes; NaN if the reference has no characters."""
    n = sum(len(r) for r in refs)
    return sum(edit_distance(r, h) for r, h in zip(refs, hyps)) / n if n else float("nan")


def region_psnr(a, b, boxes) -> float:
    mask = np.zeros(a.shape[:2], bool)
    for x0, y0, x1, y1 in boxes:
        mask[y0:y1, x0:x1] = True
    return psnr_from_mse(float(((a[mask].astype(np.float64) - b[mask]) ** 2).mean())) if mask.any() else float("nan")


In [ ]:
%%writefile src/ratflow/nn/__init__.py
"""Plain-torch ports of the pretrained networks (no diffusers dependency)."""


In [ ]:
%%writefile src/ratflow/nn/common.py
"""Building blocks shared by DC-AE and the SANA DiT, ported from diffusers 0.40.0.

Attribute names match diffusers so its state dicts load with strict=True. Numerics (dtype casts,
eps, op order) follow the diffusers code; `experiments/parity/` checks outputs against diffusers.
"""
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn


class RMSNorm(nn.Module):
    """diffusers.models.normalization.RMSNorm (non-NPU path), normalising the last dim."""

    def __init__(self, dim: int, eps: float, elementwise_affine: bool = True, bias: bool = False):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim)) if elementwise_affine else None
        self.bias = nn.Parameter(torch.zeros(dim)) if elementwise_affine and bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype
        variance = x.to(torch.float32).pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        if self.weight is None:
            return x.to(input_dtype)
        if self.weight.dtype in (torch.float16, torch.bfloat16):
            x = x.to(self.weight.dtype)
        x = x * self.weight
        if self.bias is not None:
            x = x + self.bias
        return x


def channel_rms_norm(norm: RMSNorm, x: torch.Tensor) -> torch.Tensor:
    """Apply RMSNorm over the channel dim of an NCHW tensor."""
    return norm(x.movedim(1, -1)).movedim(-1, 1)


ACTIVATIONS = {"silu": nn.SiLU, "swish": nn.SiLU, "relu": nn.ReLU, "gelu": nn.GELU, "mish": nn.Mish}


def get_activation(name: str) -> nn.Module:
    return ACTIVATIONS[name.lower()]()


class GLUMBConv(nn.Module):
    """diffusers.models.transformers.sana_transformer.GLUMBConv."""

    def __init__(self, in_channels: int, out_channels: int, expand_ratio: float = 4,
                 norm_type: str | None = None, residual_connection: bool = True):
        super().__init__()
        hidden = int(expand_ratio * in_channels)
        self.norm_type = norm_type
        self.residual_connection = residual_connection
        self.nonlinearity = nn.SiLU()
        self.conv_inverted = nn.Conv2d(in_channels, hidden * 2, 1, 1, 0)
        self.conv_depth = nn.Conv2d(hidden * 2, hidden * 2, 3, 1, 1, groups=hidden * 2)
        self.conv_point = nn.Conv2d(hidden, out_channels, 1, 1, 0, bias=False)
        self.norm = RMSNorm(out_channels, eps=1e-5, elementwise_affine=True, bias=True) if norm_type == "rms_norm" else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.nonlinearity(self.conv_inverted(x))
        x, gate = torch.chunk(self.conv_depth(x), 2, dim=1)
        x = self.conv_point(x * self.nonlinearity(gate))
        if self.norm_type == "rms_norm":
            x = channel_rms_norm(self.norm, x)
        return x + residual if self.residual_connection else x


def linear_attention(q: torch.Tensor, k_t: torch.Tensor, v: torch.Tensor, eps: float = 1e-15) -> torch.Tensor:
    """ReLU linear attention as in SANA. q, v: (..., d, N); k_t: (..., N, d). Returns (..., d, N) in fp32."""
    v = F.pad(v, (0, 0, 0, 1), mode="constant", value=1.0)  # extra row of ones -> normaliser
    out = torch.matmul(torch.matmul(v, k_t), q)
    out = out.to(torch.float32)
    return out[..., :-1, :] / (out[..., -1:, :] + eps)


In [ ]:
%%writefile src/ratflow/nn/dcae.py
"""DC-AE (Deep Compression Autoencoder) in plain torch, ported from diffusers 0.40.0
`models/autoencoders/autoencoder_dc.py`. Only what SANA's f32c32 VAE uses is kept
(ResBlock + EfficientViTBlock, rms_norm, conv downsample, interpolate upsample, tiling).

    ae = DCAE.from_pretrained("Efficient-Large-Model/Sana_1600M_1024px_diffusers", "vae")
    z = ae.encode(x)          # x in [-1, 1], NCHW; z unscaled (multiply by ae.scaling_factor for the DiT)
    x_hat = ae.decode(z)
"""
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn

from .common import GLUMBConv, RMSNorm, channel_rms_norm, get_activation, linear_attention
from .weights import load_config, load_state_dict, resolve


class ResBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, norm_type: str, act_fn: str):
        super().__init__()
        assert norm_type == "rms_norm"
        self.nonlinearity = get_activation(act_fn)
        self.conv1 = nn.Conv2d(in_channels, in_channels, 3, 1, 1)
        self.conv2 = nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False)
        self.norm = RMSNorm(out_channels, eps=1e-5, elementwise_affine=True, bias=True)

    def forward(self, x):
        h = self.conv2(self.nonlinearity(self.conv1(x)))
        return channel_rms_norm(self.norm, h) + x


class MultiscaleProjection(nn.Module):
    """diffusers SanaMultiscaleAttentionProjection."""

    def __init__(self, in_channels: int, num_heads: int, kernel_size: int):
        super().__init__()
        c = 3 * in_channels
        self.proj_in = nn.Conv2d(c, c, kernel_size, padding=kernel_size // 2, groups=c, bias=False)
        self.proj_out = nn.Conv2d(c, c, 1, 1, 0, groups=3 * num_heads, bias=False)

    def forward(self, x):
        return self.proj_out(self.proj_in(x))


class MultiscaleLinearAttention(nn.Module):
    """diffusers SanaMultiscaleLinearAttention + SanaMultiscaleAttnProcessor2_0 (rms_norm, residual)."""

    def __init__(self, channels: int, attention_head_dim: int, kernel_sizes, mult: float = 1.0, eps: float = 1e-15):
        super().__init__()
        self.eps = eps
        self.attention_head_dim = attention_head_dim
        heads = int(channels // attention_head_dim * mult)
        inner = heads * attention_head_dim
        self.to_q = nn.Linear(channels, inner, bias=False)
        self.to_k = nn.Linear(channels, inner, bias=False)
        self.to_v = nn.Linear(channels, inner, bias=False)
        self.to_qkv_multiscale = nn.ModuleList(MultiscaleProjection(inner, heads, k) for k in kernel_sizes)
        self.nonlinearity = nn.ReLU()
        self.to_out = nn.Linear(inner * (1 + len(kernel_sizes)), channels, bias=False)
        self.norm_out = RMSNorm(channels, eps=1e-5, elementwise_affine=True, bias=True)

    def forward(self, x):
        B, _, H, W = x.shape
        use_linear = H * W > self.attention_head_dim
        residual, dtype = x, x.dtype
        h = x.movedim(1, -1)
        h = torch.cat([self.to_q(h), self.to_k(h), self.to_v(h)], dim=3).movedim(-1, 1)
        h = torch.cat([h] + [blk(h) for blk in self.to_qkv_multiscale], dim=1)
        if use_linear:
            h = h.to(torch.float32)
        h = h.reshape(B, -1, 3 * self.attention_head_dim, H * W)
        q, k, v = h.chunk(3, dim=2)
        q, k = self.nonlinearity(q), self.nonlinearity(k)
        if use_linear:
            h = linear_attention(q, k.transpose(-1, -2), v, self.eps).to(dtype)
        else:  # quadratic attention for tiny maps (diffusers apply_quadratic_attention)
            s = torch.matmul(k.transpose(-1, -2), q).to(torch.float32)
            s = s / (torch.sum(s, dim=2, keepdim=True) + self.eps)
            h = torch.matmul(v, s.to(v.dtype))
        h = torch.reshape(h, (B, -1, H, W))
        h = self.to_out(h.movedim(1, -1)).movedim(-1, 1)
        return channel_rms_norm(self.norm_out, h) + residual


class EfficientViTBlock(nn.Module):
    def __init__(self, channels: int, attention_head_dim: int, qkv_multiscales):
        super().__init__()
        self.attn = MultiscaleLinearAttention(channels, attention_head_dim, qkv_multiscales)
        self.conv_out = GLUMBConv(channels, channels, norm_type="rms_norm")

    def forward(self, x):
        return self.conv_out(self.attn(x))


def _per_block(v, n):
    """Config values may be one string for all blocks or a per-block list."""
    return [v] * n if isinstance(v, str) else list(v)


def make_block(kind, channels, attention_head_dim, norm_type, act_fn, qkv_multiscales):
    if kind == "ResBlock":
        return ResBlock(channels, channels, norm_type, act_fn)
    if kind == "EfficientViTBlock":
        return EfficientViTBlock(channels, attention_head_dim, qkv_multiscales)
    raise ValueError(kind)


class DownBlock(nn.Module):
    """DCDownBlock2d with downsample=False (strided conv), i.e. downsample_block_type='Conv'."""

    def __init__(self, in_channels: int, out_channels: int, shortcut: bool = True):
        super().__init__()
        self.group_size = in_channels * 4 // out_channels
        self.shortcut = shortcut
        self.conv = nn.Conv2d(in_channels, out_channels, 3, 2, 1)

    def forward(self, x):
        h = self.conv(x)
        if self.shortcut:
            y = F.pixel_unshuffle(x, 2).unflatten(1, (-1, self.group_size)).mean(dim=2)
            h = h + y
        return h


class UpBlock(nn.Module):
    """DCUpBlock2d with interpolate=True (nearest), i.e. upsample_block_type='interpolate'."""

    def __init__(self, in_channels: int, out_channels: int, shortcut: bool = True):
        super().__init__()
        self.repeats = out_channels * 4 // in_channels
        self.shortcut = shortcut
        self.conv = nn.Conv2d(in_channels, out_channels, 3, 1, 1)

    def forward(self, x):
        h = self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))
        if self.shortcut:
            y = x.repeat_interleave(self.repeats, dim=1, output_size=x.shape[1] * self.repeats)
            h = h + F.pixel_shuffle(y, 2)
        return h


class Encoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ch, layers = cfg["encoder_block_out_channels"], cfg["encoder_layers_per_block"]
        types, ms = _per_block(cfg["encoder_block_types"], len(ch)), cfg["encoder_qkv_multiscales"]
        assert layers[0] > 0 and cfg["downsample_block_type"] == "Conv"
        self.conv_in = nn.Conv2d(cfg["in_channels"], ch[0], 3, 1, 1)
        blocks = []
        for i, (c, n) in enumerate(zip(ch, layers)):
            seq = [make_block(types[i], c, cfg["attention_head_dim"], "rms_norm", "silu", ms[i]) for _ in range(n)]
            if i < len(ch) - 1 and n > 0:
                seq.append(DownBlock(c, ch[i + 1], shortcut=True))
            blocks.append(nn.Sequential(*seq))
        self.down_blocks = nn.ModuleList(blocks)
        self.conv_out = nn.Conv2d(ch[-1], cfg["latent_channels"], 3, 1, 1)
        self.out_group = ch[-1] // cfg["latent_channels"]

    def forward(self, x):
        h = self.conv_in(x)
        for blk in self.down_blocks:
            h = blk(h)
        return self.conv_out(h) + h.unflatten(1, (-1, self.out_group)).mean(dim=2)


class Decoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ch, layers = cfg["decoder_block_out_channels"], cfg["decoder_layers_per_block"]
        types, ms = _per_block(cfg["decoder_block_types"], len(ch)), cfg["decoder_qkv_multiscales"]
        norms = _per_block(cfg.get("decoder_norm_types", "rms_norm"), len(ch))
        acts = _per_block(cfg.get("decoder_act_fns", "silu"), len(ch))
        assert layers[0] > 0 and cfg["upsample_block_type"] == "interpolate"
        self.conv_in = nn.Conv2d(cfg["latent_channels"], ch[-1], 3, 1, 1)
        self.in_repeats = ch[-1] // cfg["latent_channels"]
        up = []
        for i, (c, n) in reversed(list(enumerate(zip(ch, layers)))):
            seq = []
            if i < len(ch) - 1 and n > 0:
                seq.append(UpBlock(ch[i + 1], c, shortcut=True))
            seq += [make_block(types[i], c, cfg["attention_head_dim"], norms[i], acts[i], ms[i]) for _ in range(n)]
            up.insert(0, nn.Sequential(*seq))
        self.up_blocks = nn.ModuleList(up)
        self.norm_out = RMSNorm(ch[0], 1e-5, elementwise_affine=True, bias=True)
        self.conv_act = nn.ReLU()
        self.conv_out = nn.Conv2d(ch[0], cfg["in_channels"], 3, 1, 1)

    def forward(self, z):
        rep = z.repeat_interleave(self.in_repeats, dim=1, output_size=z.shape[1] * self.in_repeats)
        h = self.conv_in(z) + rep
        for blk in reversed(self.up_blocks):
            h = blk(h)
        return self.conv_out(self.conv_act(channel_rms_norm(self.norm_out, h)))


class DCAE(nn.Module):
    # diffusers AutoencoderDC tiling defaults (sample space)
    TILE, STRIDE = 512, 448

    def __init__(self, cfg: dict):
        super().__init__()
        self.config = dict(cfg)
        self.encoder = Encoder(cfg)
        self.decoder = Decoder(cfg)
        self.f = 2 ** (len(cfg["encoder_block_out_channels"]) - 1)
        self.scaling_factor = cfg.get("scaling_factor", 1.0)
        self.use_tiling = False

    @classmethod
    def from_pretrained(cls, repo_or_dir: str, subfolder: str | None = "vae", dtype=torch.float32):
        d = resolve(repo_or_dir, subfolder)
        with torch.device("meta"):  # no random init / no fp32 copy of the weights in RAM
            m = cls(load_config(d))
        m.load_state_dict(load_state_dict(d), strict=True, assign=True)
        return m.to(dtype).eval()

    def enable_tiling(self, on: bool = True):
        self.use_tiling = on
        return self

    # -- encode / decode (same tiling rule and blending as diffusers, so ceilings match) --
    def encode(self, x):
        if self.use_tiling and (x.shape[-1] > self.TILE or x.shape[-2] > self.TILE):
            return self._tiled_encode(x)
        return self.encoder(x)

    def decode(self, z):
        t = self.TILE // self.f
        if self.use_tiling and (z.shape[-1] > t or z.shape[-2] > t):
            return self._tiled_decode(z)
        return self.decoder(z)

    @staticmethod
    def _blend_v(a, b, n):
        n = min(a.shape[2], b.shape[2], n)
        for y in range(n):
            b[:, :, y, :] = a[:, :, -n + y, :] * (1 - y / n) + b[:, :, y, :] * (y / n)
        return b

    @staticmethod
    def _blend_h(a, b, n):
        n = min(a.shape[3], b.shape[3], n)
        for x in range(n):
            b[:, :, :, x] = a[:, :, :, -n + x] * (1 - x / n) + b[:, :, :, x] * (x / n)
        return b

    def _tiled_encode(self, x):
        f, T, S = self.f, self.TILE, self.STRIDE
        lh, lw = x.shape[2] // f, x.shape[3] // f
        tl, sl = T // f, S // f
        rows = []
        for i in range(0, x.shape[2], S):
            row = []
            for j in range(0, x.shape[3], S):
                tile = x[:, :, i:i + T, j:j + T]
                if tile.shape[2] % f or tile.shape[3] % f:
                    tile = F.pad(tile, (0, (f - tile.shape[3]) % f, 0, (f - tile.shape[2]) % f))
                row.append(self.encoder(tile))
            rows.append(row)
        return self._stitch(rows, tl - sl, sl)[:, :, :lh, :lw]

    def _tiled_decode(self, z):
        f, T, S = self.f, self.TILE, self.STRIDE
        tl, sl = T // f, S // f
        rows = [[self.decoder(z[:, :, i:i + tl, j:j + tl]) for j in range(0, z.shape[3], sl)]
                for i in range(0, z.shape[2], sl)]
        return self._stitch(rows, T - S, S)

    def _stitch(self, rows, blend, stride):
        out_rows = []
        for i, row in enumerate(rows):
            out = []
            for j, tile in enumerate(row):
                if i > 0:
                    tile = self._blend_v(rows[i - 1][j], tile, blend)
                if j > 0:
                    tile = self._blend_h(row[j - 1], tile, blend)
                out.append(tile[:, :, :stride, :stride])
            out_rows.append(torch.cat(out, dim=3))
        return torch.cat(out_rows, dim=2)


In [ ]:
%%writefile src/ratflow/nn/sana.py
"""SANA DiT (SanaTransformer2DModel) in plain torch, ported from diffusers 0.40.0
`models/transformers/sana_transformer.py`, for the configs of Sana_600M / Sana_1600M
(patch 1, no positional embedding, no guidance embedding, no qk-norm).

    dit = SanaDiT.from_pretrained("Efficient-Large-Model/Sana_1600M_1024px_diffusers", "transformer")
    v = dit(x, t, context)        # x: (B, 32, h, w) latent; t: (B,) in [0, 1000]; context: (B, L, 2304)

`expand_in_channels` adds zero-initialised input channels (e.g. to concatenate the decoded ŷ).
"""
from __future__ import annotations

import math

import torch
import torch.nn.functional as F
import torch.utils.checkpoint
from torch import nn

from .common import GLUMBConv, RMSNorm, linear_attention
from .weights import load_config, load_state_dict, resolve


def timestep_embedding(t: torch.Tensor, dim: int = 256, max_period: int = 10000) -> torch.Tensor:
    """diffusers get_timestep_embedding with flip_sin_to_cos=True, downscale_freq_shift=0."""
    half = dim // 2
    exponent = -math.log(max_period) * torch.arange(half, dtype=torch.float32, device=t.device) / half
    emb = t[:, None].float() * torch.exp(exponent)[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
    return torch.cat([emb[:, half:], emb[:, :half]], dim=-1)


class MLP2(nn.Module):
    """linear_1 -> act -> linear_2 (TimestepEmbedding / PixArtAlphaTextProjection)."""

    def __init__(self, d_in, d_hidden, act):
        super().__init__()
        self.linear_1 = nn.Linear(d_in, d_hidden)
        self.act = act
        self.linear_2 = nn.Linear(d_hidden, d_hidden)

    def forward(self, x):
        return self.linear_2(self.act(self.linear_1(x)))


class TimestepEmb(nn.Module):  # PixArtAlphaCombinedTimestepSizeEmbeddings without size conditions
    def __init__(self, dim):
        super().__init__()
        self.timestep_embedder = MLP2(256, dim, nn.SiLU())

    def forward(self, t, dtype):
        return self.timestep_embedder(timestep_embedding(t).to(dtype))


class AdaLNSingle(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.emb = TimestepEmb(dim)
        self.silu = nn.SiLU()
        self.linear = nn.Linear(dim, 6 * dim)

    def forward(self, t, dtype):
        e = self.emb(t, dtype)
        return self.linear(self.silu(e)), e


class PatchEmbed(nn.Module):
    def __init__(self, in_channels, dim, patch):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, dim, kernel_size=patch, stride=patch)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class Attention(nn.Module):
    """diffusers Attention with SanaLinearAttnProcessor2_0 (self) or SanaAttnProcessor2_0 (cross)."""

    def __init__(self, dim, heads, head_dim, bias, cross_dim=None, linear=False, out_bias=True):
        super().__init__()
        inner = heads * head_dim
        self.heads, self.linear = heads, linear
        self.to_q = nn.Linear(dim, inner, bias=bias)
        self.to_k = nn.Linear(cross_dim or dim, inner, bias=bias)
        self.to_v = nn.Linear(cross_dim or dim, inner, bias=bias)
        self.to_out = nn.ModuleList([nn.Linear(inner, dim, bias=out_bias), nn.Dropout(0.0)])

    def forward(self, x, context=None, mask=None):
        dtype = x.dtype
        ctx = x if context is None else context
        q, k, v = self.to_q(x), self.to_k(ctx), self.to_v(ctx)
        if self.linear:  # SanaLinearAttnProcessor2_0
            q = F.relu(q.transpose(1, 2).unflatten(1, (self.heads, -1)))
            k = F.relu(k.transpose(1, 2).unflatten(1, (self.heads, -1)).transpose(2, 3))
            v = v.transpose(1, 2).unflatten(1, (self.heads, -1))
            h = linear_attention(q.float(), k.float(), v.float())
            h = h.flatten(1, 2).transpose(1, 2).to(dtype)
            h = self.to_out[1](self.to_out[0](h))
            return h.clip(-65504, 65504) if dtype == torch.float16 else h
        B = x.shape[0]  # SanaAttnProcessor2_0
        hd = k.shape[-1] // self.heads
        q = q.view(B, -1, self.heads, hd).transpose(1, 2)
        k = k.view(B, -1, self.heads, hd).transpose(1, 2)
        v = v.view(B, -1, self.heads, hd).transpose(1, 2)
        if mask is not None:  # (B, L) keep-mask -> additive bias (B, 1, 1, L)
            mask = ((1 - mask.to(q.dtype)) * -10000.0)[:, None, None, :]
        h = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
        h = h.transpose(1, 2).reshape(B, -1, self.heads * hd).to(q.dtype)
        return self.to_out[1](self.to_out[0](h))


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c["num_attention_heads"] * c["attention_head_dim"]
        eps = c["norm_eps"]
        self.norm1 = nn.LayerNorm(d, elementwise_affine=False, eps=eps)
        self.attn1 = Attention(d, c["num_attention_heads"], c["attention_head_dim"], c["attention_bias"], linear=True)
        self.norm2 = nn.LayerNorm(d, elementwise_affine=c["norm_elementwise_affine"], eps=eps)
        self.attn2 = Attention(d, c["num_cross_attention_heads"], c["cross_attention_head_dim"], True,
                               cross_dim=c["cross_attention_dim"])
        self.ff = GLUMBConv(d, d, c["mlp_ratio"], norm_type=None, residual_connection=False)
        self.scale_shift_table = nn.Parameter(torch.randn(6, d) / d ** 0.5)

    def forward(self, x, context, context_mask, t6, H, W):
        B = x.shape[0]
        sh_msa, sc_msa, g_msa, sh_mlp, sc_mlp, g_mlp = (self.scale_shift_table[None] + t6.reshape(B, 6, -1)).chunk(6, dim=1)
        n = (self.norm1(x) * (1 + sc_msa) + sh_msa).to(x.dtype)
        x = x + g_msa * self.attn1(n)
        x = self.attn2(x, context, context_mask) + x
        n = self.norm2(x) * (1 + sc_mlp) + sh_mlp
        n = n.unflatten(1, (H, W)).permute(0, 3, 1, 2)
        ff = self.ff(n).flatten(2, 3).permute(0, 2, 1)
        return x + g_mlp * ff


class SanaDiT(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        c = dict(cfg)
        assert not c.get("guidance_embeds") and not c.get("qk_norm") and c.get("interpolation_scale") is None
        self.config = c
        d = c["num_attention_heads"] * c["attention_head_dim"]
        self.patch = c["patch_size"]
        self.out_channels = c.get("out_channels") or c["in_channels"]
        self.patch_embed = PatchEmbed(c["in_channels"], d, self.patch)
        self.time_embed = AdaLNSingle(d)
        self.caption_projection = MLP2(c["caption_channels"], d, nn.GELU(approximate="tanh"))
        self.caption_norm = RMSNorm(d, eps=1e-5, elementwise_affine=True)
        self.transformer_blocks = nn.ModuleList(Block(c) for _ in range(c["num_layers"]))
        self.scale_shift_table = nn.Parameter(torch.randn(2, d) / d ** 0.5)
        self.norm_out = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)  # SanaModulatedNorm.norm
        self.proj_out = nn.Linear(d, self.patch * self.patch * self.out_channels)
        self.gradient_checkpointing = False

    @classmethod
    def from_pretrained(cls, repo_or_dir: str, subfolder: str | None = "transformer", dtype=torch.float32):
        d = resolve(repo_or_dir, subfolder)
        with torch.device("meta"):  # no random init / no fp32 copy of the weights in RAM
            m = cls(load_config(d))
        m.load_state_dict(rename_legacy_keys(load_state_dict(d)), strict=True, assign=True)
        return m.to(dtype).eval()

    def forward(self, x, t, context, context_mask=None):
        B, _, h, w = x.shape
        p = self.patch
        H, W = h // p, w // p
        x = self.patch_embed(x)
        t6, t_emb = self.time_embed(t, x.dtype)
        ctx = self.caption_norm(self.caption_projection(context).view(B, -1, x.shape[-1]))
        for blk in self.transformer_blocks:
            if torch.is_grad_enabled() and self.gradient_checkpointing:
                x = torch.utils.checkpoint.checkpoint(blk, x, ctx, context_mask, t6, H, W, use_reentrant=False)
            else:
                x = blk(x, ctx, context_mask, t6, H, W)
        shift, scale = (self.scale_shift_table[None] + t_emb[:, None].to(self.scale_shift_table.device)).chunk(2, dim=1)
        x = self.proj_out(self.norm_out(x) * (1 + scale) + shift)
        x = x.reshape(B, H, W, p, p, -1).permute(0, 5, 1, 3, 2, 4)
        return x.reshape(B, -1, H * p, W * p)


LEGACY_KEYS = (  # original-SANA names still used by some Hub checkpoints (e.g. Sana_1600M_1024px_diffusers)
    ("adaln_single.", "time_embed."),
    ("pos_embed.proj.", "patch_embed.proj."),
    (".ff.inverted_conv.conv.", ".ff.conv_inverted."),
    (".ff.depth_conv.conv.", ".ff.conv_depth."),
    (".ff.point_conv.conv.", ".ff.conv_point."),
)


def rename_legacy_keys(sd: dict) -> dict:
    out = {}
    for k, v in sd.items():
        for old, new in LEGACY_KEYS:
            if old in k:
                k = k.replace(old, new)
        out[k] = v
    return out


def expand_in_channels(dit: SanaDiT, extra: int) -> SanaDiT:
    """Add `extra` zero-initialised input channels to the patch embedding (the pretrained path is unchanged)."""
    old = dit.patch_embed.proj
    new = nn.Conv2d(old.in_channels + extra, old.out_channels, old.kernel_size, old.stride,
                    device=old.weight.device, dtype=old.weight.dtype)
    with torch.no_grad():
        new.weight.zero_()
        new.weight[:, :old.in_channels] = old.weight
        new.bias.copy_(old.bias)
    dit.patch_embed.proj = new
    dit.config["in_channels"] = old.in_channels + extra
    return dit


In [ ]:
%%writefile src/ratflow/nn/weights.py
"""Load diffusers-format checkpoints without diffusers.

Sources, in order of preference:
- a local directory containing config.json and diffusion_pytorch_model{.safetensors,-0000x-of-0000y.safetensors,.pt}
- a Hugging Face repo id plus subfolder (needs huggingface_hub; used for training, not in the CLIC decoder)

The CLIC server image has torch but no safetensors, so `save_pt` writes a plain torch file for the decoder zip.
"""
from __future__ import annotations

import json
from pathlib import Path

import torch

WEIGHT_NAMES = ("diffusion_pytorch_model.safetensors", "diffusion_pytorch_model.safetensors.index.json",
                "diffusion_pytorch_model.pt", "model.pt")


def resolve(repo_or_dir: str, subfolder: str | None = None) -> Path:
    """Local directory with config + weights; downloads from the Hub if repo_or_dir is not a directory."""
    p = Path(repo_or_dir) / (subfolder or "")
    if p.is_dir():
        return p
    from huggingface_hub import snapshot_download
    pattern = f"{subfolder}/*" if subfolder else "*"
    root = snapshot_download(repo_or_dir, allow_patterns=[pattern])
    return Path(root) / (subfolder or "")


def load_config(d: Path) -> dict:
    return json.loads((Path(d) / "config.json").read_text())


def load_state_dict(d: Path) -> dict:
    d = Path(d)
    for name in WEIGHT_NAMES:
        f = d / name
        if not f.exists():
            continue
        if name.endswith(".index.json"):
            from safetensors.torch import load_file
            shards = sorted(set(json.loads(f.read_text())["weight_map"].values()))
            sd = {}
            for s in shards:
                sd.update(load_file(str(d / s)))
            return sd
        if name.endswith(".safetensors"):
            from safetensors.torch import load_file
            return load_file(str(f))
        return torch.load(f, map_location="cpu", weights_only=True)
    raise FileNotFoundError(f"no weights in {d} (looked for {WEIGHT_NAMES})")


def save_pt(model: torch.nn.Module, config: dict, d: Path, dtype: torch.dtype | None = None):
    """Write config.json + model.pt (for environments without safetensors)."""
    d = Path(d)
    d.mkdir(parents=True, exist_ok=True)
    sd = {k: (v.to(dtype) if dtype is not None and v.is_floating_point() else v).contiguous()
          for k, v in model.state_dict().items()}
    torch.save(sd, d / "model.pt")
    (d / "config.json").write_text(json.dumps(config, indent=1))


In [ ]:
%%writefile experiments/s1/train_s1.py
"""Train S1 (variable-rate latent codec) on cached DC-AE latents.

    python train_s1.py selftest                       # random model: GPU-encode / CPU-decode bit-exactness, bpp sanity
    python train_s1.py train --latents latents --out s1_out --hours 10
    (resumes from s1_out/last.pt if it exists)

Loss per sample: bpp + lambda[r] * MSE(latent_hat, latent), r ~ U{0..n_rates-1}, latents scaled by the DC-AE
scaling factor (~unit variance). bpp counts y and z bits over the crop's pixels (crop latent h*w*32*32).
"""
from __future__ import annotations

import argparse
import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "src"))
from ratflow.codec.latent_codec import LatentCodec  # noqa: E402

SCALING = 0.41407  # DC-AE f32c32 (SANA) scaling factor
# run 1 used [0.03 .. 4.0] -> Kodak 0.0155-0.12 bpp, saturating near the DC-AE ceiling above ~0.1 bpp;
# run 2 shifts down to cover ~0.010-0.08 bpp (CVPR range plus CLIC 0.075; higher rates go to the residual branch)
DEFAULT_LAMBDAS = [0.015, 0.0256, 0.0438, 0.0748, 0.128, 0.219, 0.374, 0.6]


# ---------------------------------------------------------------- data
class LatentCrops:
    """Random crops of cached full-image latents (memory-mapped)."""

    def __init__(self, root: Path, crop: int, seed: int = 0, holdout: int = 16):
        files = sorted(Path(root).glob("*.npy"))
        assert files, f"no .npy latents in {root}"
        self.arrs = [np.load(f, mmap_mode="r") for f in files]
        self.arrs = [a for a in self.arrs if min(a.shape[1:]) >= crop]
        self.val, self.arrs = self.arrs[:holdout], self.arrs[holdout:]
        self.crop = crop
        self.rng = np.random.default_rng(seed)

    def batch(self, n: int) -> torch.Tensor:
        out = np.empty((n, 32, self.crop, self.crop), np.float32)
        for i in range(n):
            a = self.arrs[self.rng.integers(len(self.arrs))]
            y = self.rng.integers(a.shape[1] - self.crop + 1)
            x = self.rng.integers(a.shape[2] - self.crop + 1)
            out[i] = a[:, y:y + self.crop, x:x + self.crop]
        return torch.from_numpy(out) * SCALING

    def val_items(self, n: int = 8):
        return [torch.from_numpy(np.asarray(a, np.float32))[None] * SCALING for a in self.val[:n]]


# ---------------------------------------------------------------- selftest
def selftest(args):
    """No training data needed: checks the codec end to end on random latents."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    m = LatentCodec().to(dev)
    m.prepare_coding()
    lat = torch.randn(1, 32, 43, 64, device=dev)  # a 1360x2048 image
    res = {}
    for r in (0, m.n_rates - 1):
        blob = m.compress(lat, r, seed=123)
        m_cpu = LatentCodec()
        m_cpu.load_state_dict(m.state_dict())  # decoder = fresh object loaded from the checkpoint, on CPU
        out_cpu = m_cpu.decompress(blob)
        out_dev = m.decompress(blob, device=dev).cpu()
        with torch.no_grad():
            est = m(lat, torch.tensor([r], device=dev))
            # what the encoder expects the decoder to reconstruct: q - u with the shared dither, through g_s
            from ratflow.entropy import dither
            x, (h, w) = m._pad(lat)
            gain = m.gain(torch.tensor([r], device=dev))
            y_r = (m.g_a(x) * gain).double().cpu().numpy()
            u = dither.u_float(dither.dither_u16(y_r.size, 123)).reshape(y_r.shape)
            y_hat = torch.from_numpy(np.round(y_r + u) - u).float().to(dev)
            expect = m.g_s(y_hat / gain)[..., :h, :w].cpu()
        bits_est = float(est["bits_y"] + est["bits_z"])
        res[r] = dict(bytes=len(blob), est_bytes=bits_est / 8, shape_ok=tuple(out_cpu.shape) == tuple(lat.shape),
                      decode_equals_encoder_view=bool(torch.equal(out_dev, expect)),
                      cpu_vs_dev_max_abs=float((out_cpu - out_dev).abs().max()))
        print(r, res[r])
    ok = all(v["shape_ok"] and v["decode_equals_encoder_view"] and v["cpu_vs_dev_max_abs"] < 1e-3
             for v in res.values())
    print("SELFTEST", "PASS" if ok else "FAIL")
    Path(args.out).mkdir(parents=True, exist_ok=True)
    (Path(args.out) / "selftest.json").write_text(json.dumps(dict(ok=ok, **{str(k): v for k, v in res.items()}), indent=1))


# ---------------------------------------------------------------- train
def train(args):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    lambdas = torch.tensor(args.lambdas, device=dev)
    data = LatentCrops(Path(args.latents), args.crop, seed=args.seed)
    m = LatentCodec(n_rates=len(args.lambdas), N=args.N, M=args.M, Z=args.Z, y_stride=args.y_stride).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=args.lr, weight_decay=0.0)
    step, log = 0, []
    ck = out / "last.pt"
    if ck.exists():
        s = torch.load(ck, map_location=dev, weights_only=False)
        m.load_state_dict(s["model"])
        opt.load_state_dict(s["opt"])
        step, log = s["step"], s["log"]
        print("resumed at step", step)
    px = (args.crop * 32) ** 2
    t_end = time.time() + args.hours * 3600
    acc = {k: torch.zeros(len(args.lambdas), device=dev) for k in ("bpp", "mse", "n")}
    while step < args.steps and time.time() < t_end:
        lr = args.lr * 0.5 * (1 + math.cos(math.pi * min(step / args.steps, 1.0)))
        for gparam in opt.param_groups:
            gparam["lr"] = lr
        lat = data.batch(args.batch).to(dev, non_blocking=True)
        r = torch.randint(0, len(args.lambdas), (args.batch,), device=dev)
        o = m(lat, r)
        bpp = (o["bits_y"] + o["bits_z"]) / px
        mse = (o["lat_hat"] - lat).pow(2).flatten(1).mean(1)
        loss = (bpp + lambdas[r] * mse).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        step += 1
        with torch.no_grad():
            acc["bpp"].index_add_(0, r, bpp.detach())
            acc["mse"].index_add_(0, r, mse.detach())
            acc["n"].index_add_(0, r, torch.ones_like(bpp))
        if step % args.log_every == 0:
            n = acc["n"].clamp_min(1)
            row = dict(step=step, lr=lr, bpp=(acc["bpp"] / n).tolist(), mse=(acc["mse"] / n).tolist())
            log.append(row)
            print(f"step {step}  lr {lr:.2e}  bpp " + " ".join(f"{v:.4f}" for v in row["bpp"])
                  + "  mse " + " ".join(f"{v:.3f}" for v in row["mse"]), flush=True)
            acc = {k: torch.zeros_like(v) for k, v in acc.items()}
        if step % args.save_every == 0:
            save(m, opt, step, log, args, out)
    save(m, opt, step, log, args, out)
    print("done at step", step)


def save(m, opt, step, log, args, out: Path):
    m.prepare_coding()
    s = dict(model=m.state_dict(), opt=opt.state_dict(), step=step, log=log, config=m.config,
             lambdas=list(args.lambdas), scaling=SCALING)
    torch.save(s, out / "last.pt.tmp")
    (out / "last.pt.tmp").replace(out / "last.pt")
    (out / "log.json").write_text(json.dumps(dict(config=m.config, lambdas=list(args.lambdas), log=log), indent=1))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["selftest", "train"])
    ap.add_argument("--latents", default="latents")
    ap.add_argument("--out", default="s1_out")
    ap.add_argument("--lambdas", type=float, nargs="+", default=DEFAULT_LAMBDAS)
    ap.add_argument("--N", type=int, default=192)
    ap.add_argument("--M", type=int, default=128)
    ap.add_argument("--Z", type=int, default=96)
    ap.add_argument("--y-stride", type=int, default=2)
    ap.add_argument("--crop", type=int, default=16, help="latent crop (16 = 512x512 pixels)")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--steps", type=int, default=200_000)
    ap.add_argument("--hours", type=float, default=10.0)
    ap.add_argument("--log-every", type=int, default=200)
    ap.add_argument("--save-every", type=int, default=5000)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    globals()[args.step](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile experiments/s1/eval_s1.py
"""End-to-end S1 evaluation with real bitstreams: image -> DC-AE -> S1 bytes -> S1 decode -> DC-AE -> image.

bpp from len(bytes) (mean over images); latent MSE; PSNR pooled over pixels (CLIC style); PSNR / MS-SSIM / LPIPS / DISTS via ratflow.eval.metrics (same code as the
ceilings). Also reports the DC-AE ceiling of each image, so the S1 points can be read against it.

    python eval_s1.py --ckpt s1_out/last.pt --dataset kodak --out s1_eval
"""
from __future__ import annotations

import argparse
import json
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "src"))
from ratflow.codec.latent_codec import LatentCodec  # noqa: E402
from ratflow.eval.metrics import Perceptual, device, images, load, pooled_psnr, save, write_csv  # noqa: E402
from ratflow.nn.dcae import DCAE  # noqa: E402

AE_REPO = "Efficient-Large-Model/Sana_1600M_1024px_diffusers"


def kodak(root: Path) -> Path:
    d = root / "kodak"
    d.mkdir(parents=True, exist_ok=True)
    for i in range(1, 25):
        f = d / f"kodim{i:02d}.png"
        if not f.exists():
            urllib.request.urlretrieve(f"http://r0k.us/graphics/kodak/kodak/kodim{i:02d}.png", f)
    return d


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default="s1_out/last.pt")
    ap.add_argument("--dataset", default="kodak", help="'kodak' or a directory of images")
    ap.add_argument("--data", type=Path, default=Path("data"))
    ap.add_argument("--out", type=Path, default=Path("s1_eval"))
    ap.add_argument("--rates", type=int, nargs="*", default=None)
    ap.add_argument("--save-images", action="store_true")
    args = ap.parse_args()

    dev = device()
    s = torch.load(args.ckpt, map_location="cpu", weights_only=False)
    m = LatentCodec(**{k: v for k, v in s["config"].items()})
    m.load_state_dict(s["model"])
    m = m.to(dev).eval()
    dec = LatentCodec(**s["config"])  # decoder side: separate object, CPU entropy path, same checkpoint
    dec.load_state_dict(s["model"])
    dec = dec.to(dev).eval()
    scaling = s.get("scaling", 0.41407)
    ae = DCAE.from_pretrained(AE_REPO, "vae").to(dev).enable_tiling()
    perc = Perceptual(dev)
    files = images(kodak(args.data) if args.dataset == "kodak" else Path(args.dataset))
    rates = args.rates if args.rates is not None else list(range(m.n_rates))
    rows = []
    args.out.mkdir(parents=True, exist_ok=True)
    for p in files:
        x = load(p)
        H, W = x.shape[:2]
        H32, W32 = H - H % 32, W - W % 32
        x = x[:H32, :W32]
        t = torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(dev)
        with torch.no_grad():
            lat = ae.encode(t) * scaling
            ceil = ae.decode(lat / scaling)
        to_img = lambda y: y[0].clamp(-1, 1).add(1).mul(127.5).round().byte().permute(1, 2, 0).cpu().numpy()  # noqa: E731
        rows.append(dict(name=p.stem, cfg="dcae_ceiling", px=H32 * W32, bpp=float("nan"), lat_mse=0.0,
                         **perc.all(x, to_img(ceil))))
        for r in rates:
            t0 = time.time()
            blob = m.compress(lat, r, seed=r)
            t_enc = time.time() - t0
            t0 = time.time()
            lat_hat = dec.decompress(blob, device=dev)
            t_dec = time.time() - t0
            with torch.no_grad():
                img = to_img(ae.decode(lat_hat / scaling))
            row = dict(name=p.stem, cfg=f"s1_r{r}", rate=r, px=H32 * W32, bytes=len(blob), bpp=len(blob) * 8 / (H32 * W32),
                       lat_mse=float((lat_hat - lat).pow(2).mean()), t_enc=t_enc, t_dec_s1=t_dec, **perc.all(x, img))
            rows.append(row)
            if args.save_images:
                save(img, args.out / "img" / f"r{r}" / f"{p.stem}.png")
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)
    write_csv(rows, args.out / "per_image.csv")

    L = [f"# S1 trên {args.dataset} (bitstream thật)", "", f"checkpoint: `{args.ckpt}` (step {s.get('step')})", "",
         "| cfg | bpp | latent MSE | PSNR (pooled) | MS-SSIM | LPIPS↓ | DISTS↓ |", "|---|---|---|---|---|---|---|"]
    summ = {}
    for cfg in ["dcae_ceiling"] + [f"s1_r{r}" for r in rates]:
        rs = [r for r in rows if r["cfg"] == cfg]
        px = [r["px"] for r in rs]
        mean = lambda k: float(np.nanmean([r[k] for r in rs]))  # noqa: E731
        summ[cfg] = dict(bpp=mean("bpp"), lat_mse=mean("lat_mse"), psnr=pooled_psnr([r["mse"] for r in rs], px),
                         msssim=mean("msssim"), lpips=mean("lpips"), dists=mean("dists"))
        v = summ[cfg]
        L.append(f"| {cfg} | {v['bpp']:.4f} | {v['lat_mse']:.4f} | {v['psnr']:.2f} | {v['msssim']:.4f} | "
                 f"{v['lpips']:.4f} | {v['dists']:.4f} |")
    L += ["", "S1 được train bằng MSE trên latent (latent_hat ≈ E[latent | bitstream]), nên ảnh mờ là đúng thiết kế; "
          "bridge S2/S3 mới là phần khôi phục độ chân thực. Dòng `dcae_ceiling` là trần của backbone."]
    (args.out / "summary.json").write_text(json.dumps(summ, indent=1))
    (args.out / "summary.md").write_text("\n".join(L), encoding="utf-8")
    print("\n".join(L))


if __name__ == "__main__":
    main()


In [ ]:
!pip install -q -U safetensors huggingface_hub lpips piq
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
import glob, os, json
cands = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/latents/*.npy', recursive=True)})
LATENTS = cands[0] if cands else 'latents'
print('latents:', LATENTS, len(glob.glob(LATENTS + '/*.npy')), 'files')
prev = glob.glob('/kaggle/input/**/s1_out/last.pt', recursive=True)  # resume from a previous run
if prev and not os.path.exists('s1_out/last.pt'):
    os.makedirs('s1_out', exist_ok=True); os.system(f'cp {prev[0]} s1_out/last.pt'); print('resume from', prev[0])

In [ ]:
!python experiments/s1/train_s1.py selftest --out s1_out
assert json.load(open('s1_out/selftest.json'))['ok'], 'SELFTEST FAIL: dừng lại, gửi log cho Claude'

In [ ]:
!python experiments/s1/train_s1.py train --latents {LATENTS} --out s1_out --hours 9.5

In [ ]:
!python experiments/s1/eval_s1.py --ckpt s1_out/last.pt --dataset kodak --out s1_eval

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('s1_eval/summary.md', encoding='utf-8').read()))

In [ ]:
!zip -qr s1_results.zip s1_out/log.json s1_out/selftest.json s1_out/last.pt s1_eval/*.md s1_eval/*.json s1_eval/*.csv && ls -la s1_results.zip